# 🛠️ Tools in LangChain

Tools are interfaces that an LLM can use to interact with the outside world.  
In this notebook, we'll cover:

1. **What are Tools?** — Concept & structure
2. **Creating Tools** — `@tool` decorator, `StructuredTool`, `Tool` class
3. **Binding Tools to a Model** — `bind_tools()`
4. **Tool Calling** — How LLM decides to call a tool
5. **Executing Tool Calls** — Running the tool and returning results
6. **Building a Simple Agent** — Full loop with tools

---
## 📖 LangChain Tools — Complete Theory (Infographic)

![LangChain Tools Theory](langchain_tools_theory.jpg)

---

### 🔍 Theory Explanation

**Tools kya hain?**  
Tools ek **interface** hain jo LLM ko bahar ki duniya se interact karne dete hain. Normally ek LLM sirf text generate karta hai — lekin tools ki madad se woh **calculations** kar sakta hai, **APIs** call kar sakta hai, **databases** query kar sakta hai, aur bahut kuch.

Har tool ke **4 components** hote hain:
- **Name** — Tool ka unique naam (e.g., `add`, `multiply`)
- **Description** — LLM isse padhta hai aur decide karta hai kab use karna hai
- **Input Schema** — Kya inputs chahiye (type hints / Pydantic model se banta hai)
- **Function** — Actual logic jo execute hota hai

---

### 🏗️ Tools Banane ke 3 Tarike

| Method | Kab Use Karein | Key Point |
|---|---|---|
| `@tool` Decorator | Simple functions ke liye | Docstring = Description, Type hints = Schema |
| `StructuredTool` | Custom name/description chahiye | Pydantic schema se full control |
| `BaseTool` Class | Complex tools ke liye | Inherit karke `_run()` method define karo |

---

### 🔄 Tool Calling Flow (Kaise Kaam Karta Hai)

```
User Query → LLM (bind_tools) → tool_calls response → Execute Tool → ToolMessage → LLM → Final Answer
```

1. **User** apna question bhejta hai
2. **LLM** decide karta hai ki kaunsa tool call karna hai (`tool_calls` return karta hai)
3. **Hum** tool ko execute karte hain (LLM khud execute NAHI karta!)
4. **Result** ko `ToolMessage` mein wrap karke wapas LLM ko bhejte hain
5. **LLM** final human-readable answer deta hai

> ⚠️ **Important:** LLM sirf **SUGGEST** karta hai ki kaunsa tool call karna hai. Actually execute karna **hamari** responsibility hai!

---

### 🧠 Key Concepts

- **`bind_tools()`** — Model ko batata hai ki kaunse tools available hain
- **`tool_calls`** — Model ka response jismein tool name + arguments hote hain
- **`ToolMessage`** — Tool ka result wapas model ko bhejna (`tool_call_id` zaroori hai!)
- **Agent Loop** — Ye poora cycle repeat hota hai jab tak final answer nahi aa jaata

## 🔧 Setup — Load API Keys & Initialize Model

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq

# Initialize Groq Model
model = ChatGroq(model='openai/gpt-oss-20b')
print('Model initialized successfully!')

---
## 1️⃣ Creating Tools using `@tool` Decorator

The simplest way to create a tool. The function's **docstring** becomes the tool description that the LLM reads to decide when to use it.

In [ ]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b


# Inspect the tool
print('Tool Name:', add.name)
print('Tool Description:', add.description)
print('Tool Args Schema:', add.args_schema.model_json_schema())
print()

# Directly invoke the tool
result = add.invoke({'a': 5, 'b': 3})
print('add(5, 3) =', result)

---
## 2️⃣ Creating Tools using `StructuredTool`

More control over name, description, and schema. Useful when you want to customize beyond what the decorator offers.

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field


# Define input schema with Pydantic
class CalculatorInput(BaseModel):
    a: float = Field(description='First number')
    b: float = Field(description='Second number')


def power_function(a: float, b: float) -> float:
    """Raise first number to the power of second number."""
    return a ** b


# Create StructuredTool
power_tool = StructuredTool.from_function(
    func=power_function,
    name='power',
    description='Raise the first number to the power of the second number. Use this for exponentiation.',
    args_schema=CalculatorInput
)

print('Tool Name:', power_tool.name)
print('Tool Description:', power_tool.description)
print('Tool Args:', power_tool.args_schema.model_json_schema())
print()

# Test it
result = power_tool.invoke({'a': 2, 'b': 10})
print('power(2, 10) =', result)

---
## 3️⃣ Creating a More Practical Tool

Let's create a tool that does something more useful — like getting the current time or searching for information.

In [ ]:
from datetime import datetime


@tool
def get_current_time() -> str:
    """Get the current date and time. Use this when the user asks about today's date or current time."""
    now = datetime.now()
    return now.strftime('%Y-%m-%d %H:%M:%S (%A)')


@tool
def calculate_age(birth_year: int) -> str:
    """Calculate someone's age given their birth year."""
    current_year = datetime.now().year
    age = current_year - birth_year
    return f'A person born in {birth_year} is approximately {age} years old.'


@tool
def word_count(text: str) -> int:
    """Count the number of words in a given text."""
    return len(text.split())


# Test
print(get_current_time.invoke({}))
print(calculate_age.invoke({'birth_year': 2000}))
print('Word count:', word_count.invoke({'text': 'LangChain tools are awesome'}))

---
## 4️⃣ Binding Tools to the Model — `bind_tools()`

This tells the LLM what tools are available. The model can then decide **when** and **which** tool to call based on the user's query.

In [ ]:
# Collect all our tools
tools = [add, multiply, power_tool, get_current_time, calculate_age, word_count]

# Bind tools to the model
model_with_tools = model.bind_tools(tools)

print(f'Model now has access to {len(tools)} tools:')
for t in tools:
    print(f'  - {t.name}: {t.description}')

---
## 5️⃣ Tool Calling — LLM Decides Which Tool to Use

When you send a message to the model with tools bound, the model returns `tool_calls` in its response instead of (or along with) a text answer.

In [ ]:
from langchain_core.messages import HumanMessage

# Ask something that requires a tool
response = model_with_tools.invoke([HumanMessage(content='What is 15 multiplied by 23?')])

print('Content:', response.content)
print()
print('Tool Calls:', response.tool_calls)

In [ ]:
# Ask about time — should trigger get_current_time tool
response2 = model_with_tools.invoke([HumanMessage(content='What is the current date and time?')])

print('Content:', response2.content)
print()
print('Tool Calls:', response2.tool_calls)

In [ ]:
# Ask something that does NOT need any tool
response3 = model_with_tools.invoke([HumanMessage(content='What is the capital of India?')])

print('Content:', response3.content)
print()
print('Tool Calls:', response3.tool_calls)
print()
print('👆 No tool calls — model answered directly!')

---
## 6️⃣ Executing Tool Calls Manually

The model only **suggests** which tool to call. We need to actually **execute** it and feed the result back.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# Create a tool name -> tool function mapping
tool_map = {t.name: t for t in tools}

# Step 1: Send user message
user_msg = HumanMessage(content='What is 2 raised to the power of 8?')
ai_response = model_with_tools.invoke([user_msg])

print('AI Response tool_calls:', ai_response.tool_calls)
print()

# Step 2: Execute each tool call
tool_messages = []
for tool_call in ai_response.tool_calls:
    tool_name = tool_call['name']
    tool_args = tool_call['args']
    tool_id = tool_call['id']
    
    print(f'Calling tool: {tool_name}({tool_args})')
    
    # Execute the tool
    result = tool_map[tool_name].invoke(tool_args)
    print(f'Result: {result}')
    
    # Create ToolMessage with the result
    tool_messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))

# Step 3: Send everything back to the model for final answer
full_conversation = [user_msg, ai_response] + tool_messages
final_response = model_with_tools.invoke(full_conversation)

print()
print('✅ Final Answer:', final_response.content)

---
## 7️⃣ Building a Simple Agent Loop

An **agent** is just a loop that:
1. Sends user query to LLM
2. If LLM returns tool calls → execute them → feed results back
3. Repeat until LLM gives a final text answer

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage


def run_agent(query: str, model_with_tools, tool_map, max_iterations=5):
    """Simple agent loop that handles tool calls."""
    print(f'🧑 User: {query}')
    print('=' * 60)
    
    messages = [HumanMessage(content=query)]
    
    for i in range(max_iterations):
        # Get model response
        response = model_with_tools.invoke(messages)
        messages.append(response)
        
        # If no tool calls, we have our final answer
        if not response.tool_calls:
            print(f'\n🤖 Agent: {response.content}')
            return response.content
        
        # Execute each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call['id']
            
            print(f'  🔧 Calling: {tool_name}({tool_args})')
            
            result = tool_map[tool_name].invoke(tool_args)
            print(f'  📤 Result: {result}')
            
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))
    
    return 'Max iterations reached.'


# Test the agent!
run_agent('What is 25 multiplied by 40, and also tell me the current time?', model_with_tools, tool_map)

In [ ]:
# Another test — age calculation
run_agent('I was born in 1998. How old am I?', model_with_tools, tool_map)

In [ ]:
# Test — no tool needed
run_agent('What is LangChain?', model_with_tools, tool_map)

---
## 8️⃣ Using LangChain's Built-in Agent (create_tool_calling_agent)

Instead of writing the loop ourselves, LangChain provides `create_tool_calling_agent` + `AgentExecutor`.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Use the available tools to help answer questions accurately.'),
    ('placeholder', '{chat_history}'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}')
])

# Create agent
agent = create_tool_calling_agent(model, tools, prompt)

# Create executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print('Agent created successfully!')

In [ ]:
# Test the agent executor
result = agent_executor.invoke({'input': 'What is 5 added to 10, then multiply the result by 3?'})
print()
print('Final Output:', result['output'])

In [ ]:
# Another test
result = agent_executor.invoke({'input': 'Tell me the current time and also count the words in: LangChain is a framework for building LLM applications'})
print()
print('Final Output:', result['output'])

---
## 📝 Summary

| Concept | Description |
|---|---|
| `@tool` decorator | Simplest way to create a tool from a function |
| `StructuredTool` | More control with custom name, description, schema |
| `bind_tools()` | Tells the model what tools are available |
| `tool_calls` | Model's response containing which tool to call |
| `ToolMessage` | Feeds tool execution results back to the model |
| Agent Loop | Repeated cycle of: query → tool call → execute → respond |
| `AgentExecutor` | LangChain's built-in agent loop handler |

### Key Points:
- The **docstring** of a tool is critical — it's what the LLM reads to decide when to use the tool
- The model **suggests** tool calls, but **we** execute them
- `ToolMessage` must include the `tool_call_id` to match the request
- An agent is simply a **loop** around tool calling